# 第 4 章 夫琅禾费衍射与透镜 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/waveprop/ch4/`。

## 夫琅禾费衍射

`ch4_fraunhofer.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 4 章 =====
夫琅禾费衍射（原书 4.1 节，Listings 4.1-4.2 与图 4.1）

原书要点：
    夫琅禾费衍射积分（式 4.1）在 Delta_z > 2 D^2 / lambda（式 4.2，远场判据）时
    成立，它可以写成一次二维 FT（式 4.3），观察面坐标与源面空间频率按
        x2 = lambda*Delta_z*fx1,  y2 = lambda*Delta_z*fy1   （式 4.4）
    线性映射。数值实现 = 一次定标正确的 ft2（Listing 4.1 fraunhofer_prop）。

数值验证（运行本脚本全部自动打印）：
    (0) jinc 约定核对：原书解析式用 jinc(D*r2/(lambda*Dz))，其中
        jinc(u) = 2 J1(pi u)/(pi u)（u=0 处为 1，第一个零点在 u=1.22，
        即爱里环半径 1.22*lambda*Dz/D）——与 base.py 圆盘谱 jinc 的关系；
    (1) 书例复现（N=512, L=7.5 mm, D=1 mm, lambda=1 um, Dz=20 m）：
        轴上辐照度、传感器区域（±0.2 m）内数值 vs 解析的最大偏差；
    (2) 能量守恒（离散 Parseval）：sum|U2|^2*d2^2 == 源面孔径面积
        （入射单位振幅平面波），并对比解析场在观察窗内的真实能量
        —— 差值即被"卷回"窗内的混叠能量；
    (3) 采样不足的混叠表现：N 从 512 减到 64（d1 变粗 -> 观察窗
        x2max = lambda*Dz/(2*d1) 变窄），图样比窗口宽时边缘卷回；
    (4) 两条路线对照：直接按菲涅耳衍射积分（式 4.6）做二维数值求和
        （不用 FFT）与 fraunhofer_prop 的 FT 路线逐点对比。

MATLAB/Python 差异（对应 .m 文件内注释）：
    - MATLAB 数组 1 起、numpy 0 起：居中排列的零坐标样本位置不同，
      频率轴一律 fX = (arange(N) - N//2)/(N*d1) 生成（与原书
      (-N/2 : N/2-1)/(N*d1) 相同的升序居中排列）。
    - np.meshgrid 默认 indexing='xy'，与 MATLAB meshgrid 一致：
      x2 沿列变化、y2 沿行变化。
    - 原书 ft2 = fftshift(fft2(fftshift(g)))*delta^2（Listing 2.5）；
      base.py 的 ft2_centered 对输入先做 ifftshift——偶数 N 时
      fftshift 与 ifftshift 相同，两者逐元素一致。
用法：python ch4_fraunhofer.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib + scipy（J0/J1，能量与 jinc 核对用）；
      base.py（同仓库 python/fourier/common，circle/jinc/ft2_centered）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j0, j1

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ---- 复用《计算傅里叶光学》笔记的公共基础模块（同仓库） ----
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import circle, jinc, ft2_centered  # noqa: E402


# ===== 书 Listing 4.1：fraunhofer_prop =====
def fraunhofer_prop(Uin, wvl, d1, Dz):
    """夫琅禾费传播（原书 Listing 4.1）。

    Uout = exp(i k/(2 Dz) (x2^2+y2^2)) / (i lambda Dz) * ft2(Uin, d1)
    二次相位因子只留在积分外（积分内的已被远场近似扔掉）；
    e^{i k Dz} 轴上相位与原书一样不计算。
    返回 Uout（居中排列）、x2、y2（升序坐标，单位 m）。
    """
    N = Uin.shape[0]                       # assume square grid
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)   # 源面频率轴（升序、居中排列）
    x2, y2 = np.meshgrid(wvl * Dz * fX, wvl * Dz * fX)  # 式 (4.4) 的线性映射
    Uout = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * ft2_centered(Uin, d1)
    return Uout, x2, y2


def jinc_s(u):
    """原书解析式里的 jinc：jinc(u) = 2 J1(pi u)/(pi u)，jinc(0)=1。"""
    u = np.asarray(u, dtype=float)
    out = np.ones_like(u)
    nz = u != 0
    out[nz] = 2.0 * j1(np.pi * u[nz]) / (np.pi * u[nz])
    return out


def circ(X, Y, D):
    """直径 D 的圆孔（原书 circ(x,y,D) 约定，|r| <= D/2 取 1）。"""
    return circle(X, Y, D / 2.0)


def main(outdir=None):
    # ---- 原书 Listing 4.2 的参数 ----
    N = 512          # number of grid points per side
    L = 7.5e-3       # total size of the grid [m]
    d1 = L / N       # source-plane grid spacing [m]
    D = 1e-3         # diameter of the aperture [m]
    wvl = 1e-6       # optical wavelength [m]
    k = 2 * np.pi / wvl
    Dz = 20.0        # propagation distance [m]

    area = np.pi * D ** 2 / 4.0          # 圆孔面积 = FT 的直流值
    x1, y1 = np.meshgrid((np.arange(N) - N // 2) * d1,
                         (np.arange(N) - N // 2) * d1)
    Uin = circ(x1, y1, D)

    print('===== 验证 0：jinc 约定核对 =====')
    rho = np.linspace(0, 400, 4001)      # rho = r2/(lambda*Dz)
    diff = np.max(np.abs(jinc(rho, R=D / 2) - area * jinc_s(D * rho)))
    print('    max|base.jinc(rho, R=D/2) - 面积*jinc_s(D*rho)| = %.2e'
          % diff)
    print('    （即 base.jinc(rho,R=D/2) = 面积 × 2J1(pi D rho)/(pi D rho)，'
          '爱里图样）')
    u_first_zero = 3.8317 / np.pi
    print('    jinc_s 第一个零点 u = %.4f -> 爱里环半径 1.22*lambda*Dz/D = %.4f m'
          % (u_first_zero, u_first_zero * wvl * Dz / D))

    print('===== 验证 1 + 2：书例复现（N=512）与能量守恒 =====')
    Uout, x2, y2 = fraunhofer_prop(Uin, wvl, d1, Dz)
    d2 = wvl * Dz / L                    # 观察面采样间隔 = lambda*Dz*delta_f1
    x2max = wvl * Dz / (2 * d1)          # 观察窗半宽 = lambda*Dz*f1_max
    print('    d1 = %.3e m, d2 = lambda*Dz/L = %.4f m, 观察窗 ±%.3f m'
          % (d1, d2, x2max))
    U_th = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * area * jinc_s(D * np.hypot(x2, y2) / (wvl * Dz))
    I_num, I_ana = np.abs(Uout) ** 2, np.abs(U_th) ** 2
    peak = (area / (wvl * Dz)) ** 2
    print('    轴上辐照度：数值 %.6f mW/m^2 vs 解析 %.6f mW/m^2（高 %.2f%%）'
          % (I_num[N // 2, N // 2] * 1e3, peak * 1e3,
             100 * (I_num[N // 2, N // 2] / peak - 1)))
    # 两类误差分开看：
    #   标度误差 = 数值整体比解析高（节点采样把孔径边界样本整格计入，
    #              直流样本偏大 -> 各处等比例抬高，见《计算傅里叶光学》
    #              笔记 ch2 的"胞元中心采样"坑）；
    #   形状误差 = 各自归一化后的残差（这才反映混叠/相位误差）。
    shape = np.abs(I_num / I_num.max() - I_ana / I_ana.max())
    sensor = np.abs(x2) <= 0.2           # 原书"0.4 m 宽的靶板传感器"
    print('    传感器区域（±0.2 m）：标度误差 %.2e，归一化形状误差 %.2e'
          % (np.max(np.abs(I_num - I_ana)[sensor]) / peak, shape[sensor].max()))
    imax = np.unravel_index(shape.argmax(), shape.shape)
    print('    形状误差最大值 %.2e 在 x2=%.3f m（第一爱里零点 0.0244 m 附近的陡坡）'
          % (shape.max(), x2[imax]))
    print('    边缘（|x2|>0.6 m）形状误差只有 %.1e——但那里的图样本身只有峰值的'
          ' 1e-6 倍，相对局部信号仍显著（原书"大范围显示时边缘开始出偏差"）'
          % shape[np.abs(x2) > 0.6].max())

    P_num = np.sum(I_num) * d2 ** 2      # 离散 Parseval：应恒等于孔径面积
    print('    能量：sum|U2|^2*d2^2 = %.6e（孔径面积 %.6e，相对偏差 %.2e）'
          % (P_num, area, abs(P_num - area) / area))
    # 解析场在观察窗内的真实能量（精细网格上数值积分）
    nf = 1501
    xf = np.linspace(-x2max, x2max, nf)
    Xf, Yf = np.meshgrid(xf, xf)
    I_fine = ((area / (wvl * Dz)) ** 2) * \
        jinc_s(D * np.hypot(Xf, Yf) / (wvl * Dz)) ** 2
    P_ana_win = np.trapezoid(np.trapezoid(I_fine, xf), xf)
    print('    解析场窗内能量 = %.6e（占总能量 %.2f%%）-> 混叠卷回约 %.2f%%'
          % (P_ana_win, 100 * P_ana_win / area, 100 * (1 - P_ana_win / area)))

    print('===== 验证 3：采样不足的混叠（N=64，观察窗变窄） =====')
    N2 = 64
    d1b = L / N2
    x1b, y1b = np.meshgrid((np.arange(N2) - N2 // 2) * d1b,
                           (np.arange(N2) - N2 // 2) * d1b)
    Uout_b, x2b, y2b = fraunhofer_prop(circ(x1b, y1b, D), wvl, d1b, Dz)
    d2b = wvl * Dz / L
    U_th_b = np.exp(1j * k / (2 * Dz) * (x2b ** 2 + y2b ** 2)) \
        / (1j * wvl * Dz) * area * \
        jinc_s(D * np.hypot(x2b, y2b) / (wvl * Dz))
    I_num_b, I_ana_b = np.abs(Uout_b) ** 2, np.abs(U_th_b) ** 2
    shape_b = np.abs(I_num_b / I_num_b.max() - I_ana_b / I_ana_b.max())
    print('    观察窗 ±%.4f m（爱里环半径 0.0244 m -> 窗内只有 ~3.5 个环）'
          % (wvl * Dz / (2 * d1b)))
    print('    归一化形状误差 max = %.3f（外环整个错位；标度误差另计 %.2f%%）'
          % (shape_b.max(), 100 * (I_num_b.max() / I_ana_b.max() - 1)))
    P_num_b = np.sum(I_num_b) * d2b ** 2
    xf_b = np.linspace(-wvl * Dz / (2 * d1b), wvl * Dz / (2 * d1b), 2001)
    Xfb, Yfb = np.meshgrid(xf_b, xf_b)
    I_fb = ((area / (wvl * Dz)) ** 2) * \
        jinc_s(D * np.hypot(Xfb, Yfb) / (wvl * Dz)) ** 2
    P_ana_b = np.trapezoid(np.trapezoid(I_fb, xf_b), xf_b)
    print('    能量账本：数值 sum|U2|^2*d2^2 = %.6e（仍=孔径面积！混叠不丢能量，'
          '只搬能量）' % P_num_b)
    print('              解析窗内真能量 = %.6e（占 %.2f%%）-> 卷回 %.2f%%'
          % (P_ana_b, 100 * P_ana_b / area, 100 * (1 - P_ana_b / area)))

    print('===== 验证 4：两条路线对照（直接积分 vs FT） =====')
    # 直接按菲涅耳衍射积分（式 4.6）做二维求和（y2=0 切片），不用 FFT：
    # U(x2,0) = 1/(i lambda Dz) * sum U1 * exp(i k/2Dz (x1^2+y1^2))
    #           * exp(-i k/Dz x1 x2) * d1^2
    # 只对孔径内的样本求和（孔径外 U1=0）。
    inside = Uin != 0
    xa, ya = x1[inside], y1[inside]
    quad = np.exp(1j * k / (2 * Dz) * (xa ** 2 + ya ** 2))   # 被远场近似扔掉的相位
    xs = x2[N // 2, :]                       # y2=0 行的 x2（升序）
    integ = np.einsum('i,ji->j', quad,
                      np.exp(-1j * k / Dz * xs[:, None] * xa[None, :]))
    U_dir = integ * d1 ** 2 / (1j * wvl * Dz)
    U_dir *= np.exp(1j * k / (2 * Dz) * xs ** 2)
    sl = N // 2
    diff_direct = np.max(np.abs(np.abs(U_dir) - np.abs(Uout[sl, :]))) \
        / np.max(np.abs(Uout[sl, :]))
    print('    y2=0 切片：max||U_direct|-|U_ft||/max|U_ft| = %.2e'
          % diff_direct)
    print('    （远场近似丢弃的二次相位在孔径边缘只有 %.3f rad，'
          '所以两条路线几乎重合）'
          % (k / (2 * Dz) * (D / 2) ** 2))

    if outdir:
        # ---- 图 fig-4-5：书例复现（原书图 4.1 的自绘版 + 二维图样） ----
        fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
        half = int(0.1 / d2) + 1
        sub = I_num[N // 2 - half:N // 2 + half, N // 2 - half:N // 2 + half]
        im = axes[0].imshow(sub,
                            extent=[-half * d2, half * d2, -half * d2, half * d2],
                            cmap='inferno', origin='lower')
        axes[0].set_title('(a) 观察面辐照度（±0.1 m，线性标度）')
        axes[0].set_xlabel(r'$x_2$ [m]')
        axes[0].set_ylabel(r'$y_2$ [m]')
        fig.colorbar(im, ax=axes[0], shrink=0.85)
        sl = slice(N // 2 - half, N // 2 + half)
        axes[1].plot(x2[N // 2, sl], I_ana[N // 2, sl],
                     color='0.6', lw=1.8, ls='--', label='解析')
        axes[1].plot(x2[N // 2, sl], I_num[N // 2, sl], 'b.', ms=4,
                     label='数值')
        axes[1].set_xlabel(r'$x_2$ [m]')
        axes[1].set_ylabel(r'辐照度 [W/m$^2$]')
        axes[1].set_title('(b) $y_2=0$ 切片：数值 vs 解析（原书图 4.1 复现）')
        axes[1].legend(fontsize=9)
        axes[2].plot(x2[N // 2, sl], shape[N // 2, sl], 'k.')
        axes[2].set_xlabel(r'$x_2$ [m]')
        axes[2].set_ylabel(r'归一化形状误差')
        axes[2].set_title('(c) 残差在爱里环陡坡处最大（直流标度差已扣除）')
        for ax in axes:
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fp = os.path.join(outdir, 'fig-4-5.png')
        fig.savefig(fp, dpi=150)
        print('\n已保存：%s' % fp)

        # ---- 图 fig-4-6：采样不足的混叠 ----
        fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
        hb = N2 // 2
        axes[0].plot(x2b[hb, :], I_ana_b[hb, :], color='0.6', lw=1.8,
                     ls='--', label='解析')
        axes[0].plot(x2b[hb, :], I_num_b[hb, :], 'b.', ms=4, label='数值（N=64）')
        axes[0].axvline(wvl * Dz / (2 * d1b), color='r', ls=':', lw=1)
        axes[0].axvline(-wvl * Dz / (2 * d1b), color='r', ls=':', lw=1)
        axes[0].set_xlabel(r'$x_2$ [m]')
        axes[0].set_ylabel(r'辐照度 [W/m$^2$]')
        axes[0].set_title('(a) N=64：观察窗 ±0.085 m，图样被卷回窗内')
        axes[0].legend(fontsize=9)
        half5 = int(0.1 / d2) + 1
        sl5 = slice(N // 2 - half5, N // 2 + half5)
        axes[1].semilogy(np.abs(x2[N // 2, sl5]), shape[N // 2, sl5] + 1e-16,
                         'k.', ms=3, label='N=512（窗 ±0.68 m）')
        xb_c = np.abs(x2b[hb, :])
        axes[1].semilogy(xb_c, shape_b[hb, :] + 1e-16, 'r.', ms=4,
                         label='N=64（窗 ±0.085 m）')
        axes[1].set_xlabel(r'$|x_2|$ [m]')
        axes[1].set_ylabel(r'归一化形状误差')
        axes[1].set_title('(b) 同一位置的误差：N=64（红）比 N=512（黑）大两个量级')
        axes[1].legend(fontsize=9)
        for ax in axes:
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fp = os.path.join(outdir, 'fig-4-6.png')
        fig.savefig(fp, dpi=150)
        print('已保存：%s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## lenses

`ch4_lenses.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 4 章 =====
透镜的傅里叶变换性质（原书 4.2 节，Listings 4.3-4.5 与图 4.2-4.4）

原书要点：焦距 f_l 的会聚薄镜给场乘相位 exp(-i k/(2 f_l)(x^2+y^2))
（式 4.7）。透明物片放在三个位置、传播到后焦面（Delta_z = f_l）时，
物面上的透镜相位与菲涅耳积分（式 4.6）的二次相位相消，剩下的都是
"常数 prefactor x 二次相位 x 一个 FT"，只是 FT 的频率标度与相位因子不同：
    物贴透镜        式 (4.10)：pref = 1/(i lambda f_l)，相位 k/(2 f_l) r2^2
    物在透镜前 d    式 (4.12)：pref 同上，相位 k/(2 f_l)(1-d/f_l) r2^2，
                    且严格形式带移位光瞳 P(x1 + d/f_l x2)（渐晕，Listing 4.4 未实现）
    物在透镜后 d    式 (4.16)：pref = f_l/d / (i lambda d)，相位 k/(2 d) r2^2，
                    FT 标度 lambda d（几何光学近似，要求 d << f_l）
两个特例：d=0 退回物贴透镜；d=f_l（物在前焦面）时相位因子=1，成为精确 FT。

原书勘误（本脚本按修正后的签名实现）：
    Listing 4.5 第 1-2 行的函数签名 lens_behind_ft(Uin, wvl, d1, f) 少了
    参数 d，而函数体（第 10、16-17 行）用到 d；第 4 行注释里的
    lens_behind_ft(Uin, wvl, d1, d, f) 才是正确签名。另外 Listing 4.4/4.5
    的返回值顺序 [x2 y2 Uout] 与 Listing 4.1/4.3 的 [Uout x2 y2] 不一致
    （能跑，但混用容易翻车）。

数值验证（运行本脚本全部自动打印）：
    (1) d=0 时 lens_in_front_ft 与 lens_against_ft 逐点一致；
    (2) d=f_l 时 lens_in_front_ft 退化为精确 FT（无二次相位），并与
        物贴透镜结果相差恰好 exp(i k/(2 f_l) r2^2)；
    (3) 独立路线对照：直接按菲涅耳衍射积分（式 4.6）对"物面场 =
        透过率 x 透镜相位"做二维数值求和（不用 FFT），与
        lens_against_ft 的 FT 路线逐点对比；
    (4) 能量守恒（离散 Parseval）：sum|U2|^2 d2^2 == 物面积/(lambda f_l)^2；
    (5) 物在透镜后：焦斑宽度 ~ lambda d / D 与峰值 ~ (f_l/d)^2 的实测 vs 理论；
    (6) 渐晕演示：物前 d=0.5 f_l、有限光瞳时，式 (4.11) 的移位光瞳
        P(x1 + d/f_l x2) 使离轴焦点变暗——Listing 4.4（无移位）高估边缘。

MATLAB/Python 差异：
    - 原书 ft2 = fftshift(fft2(fftshift(g)))*delta^2；base.py 的
      ft2_centered 输入端用 ifftshift，偶数 N 下两者逐元素一致。
    - np.meshgrid 默认 'xy'，与 MATLAB meshgrid 一致（x2 沿列变化）。
用法：python ch4_lenses.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib；base.py（同仓库 python/fourier/common）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import rect, ft2_centered  # noqa: E402


# ===== 书 Listing 4.3：物贴透镜（lens_against_ft） =====
def lens_against_ft(Uin, wvl, d1, f):
    """物贴透镜 -> 后焦面（式 4.10）。f = f_l 焦距 [m]。"""
    N = Uin.shape[0]
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)
    x2, y2 = np.meshgrid(wvl * f * fX, wvl * f * fX)   # 观察面坐标：标度 lambda*f_l
    Uout = np.exp(1j * k / (2 * f) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * f) * ft2_centered(Uin, d1)
    return Uout, x2, y2


# ===== 书 Listing 4.4：物在透镜前（lens_in_front_ft） =====
def lens_in_front_ft(Uin, wvl, d1, f, d):
    """物在透镜前 d -> 后焦面（式 4.12，取 P=1，即不含移位光瞳的渐晕项）。

    d=0 退化为 lens_against_ft；d=f 时二次相位消失（精确 FT）。
    注意：原书 Listing 4.4 的返回值顺序是 [x2 y2 Uout]，这里统一为
    (Uout, x2, y2)，与其余两个函数保持一致。"""
    N = Uin.shape[0]
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)
    x2, y2 = np.meshgrid(wvl * f * fX, wvl * f * fX)
    Uout = 1 / (1j * wvl * f) \
        * np.exp(1j * k / (2 * f) * (1 - d / f) * (x2 ** 2 + y2 ** 2)) \
        * ft2_centered(Uin, d1)
    return Uout, x2, y2


# ===== 书 Listing 4.5：物在透镜后（lens_behind_ft，勘误修正签名） =====
def lens_behind_ft(Uin, wvl, d1, d, f):
    """物在透镜后（距焦面 d）-> 焦面（式 4.16）。

    原书 Listing 4.5 的签名缺 d（勘误），这里补上；FT 标度是 lambda*d，
    幅度乘 f_l/d（会聚球面波在物面处的振幅，式 4.13）。"""
    N = Uin.shape[0]
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)
    x2, y2 = np.meshgrid(wvl * d * fX, wvl * d * fX)   # 标度是 lambda*d ！
    Uout = f / d * 1 / (1j * wvl * d) \
        * np.exp(1j * k / (2 * d) * (x2 ** 2 + y2 ** 2)) \
        * ft2_centered(Uin, d1)
    return Uout, x2, y2


def fresnel_direct_slice(U1, wvl, d1, Dz, x2_vals, y2=0.0):
    """独立路线：按菲涅耳衍射积分（式 4.6）对全部网格点直接二维求和
    （不用 FFT），返回 y=y2 行各 x2_vals 处的 U。

    U(x2,y2) = e^{ik Dz}/(i lambda Dz) sum U1(x1,y1)
               x e^{i k/2Dz (x1^2+y1^2)} e^{-i 2pi/(lambda Dz)(x2 x1 + y2 y1)} d1^2
    （e^{ik Dz} 轴上相位与原书一样略去；求和点数 = N^2，演示用小网格。）"""
    N = U1.shape[0]
    k = 2 * np.pi / wvl
    n = np.arange(N) - N // 2
    xa = n * d1
    X1, Y1 = np.meshgrid(xa, xa)
    quad = np.exp(1j * k / (2 * Dz) * (X1 ** 2 + Y1 ** 2))    # 源面二次相位
    kern = np.exp(-2j * np.pi / (wvl * Dz)
                  * (x2_vals[:, None] * X1.ravel()[None, :]
                     + y2 * Y1.ravel()[None, :]))
    src = (U1 * quad).ravel()
    return np.einsum('j,ij->i', src, kern) * d1 ** 2 / (1j * wvl * Dz)


def main(outdir=None):
    # ---- 演示参数：0.5 mm 方孔物片，f_l = 1 m，lambda = 1 um ----
    N = 256
    L = 2.5e-3         # 源面网格宽度 [m]
    d1 = L / N
    D = 0.5e-3         # 方孔边长 [m]
    wvl = 1e-6
    k = 2 * np.pi / wvl
    fl = 1.0           # 焦距 [m]
    n = np.arange(N) - N // 2
    x1, y1 = np.meshgrid(n * d1, n * d1)
    tA = rect(x1 / D) * rect(y1 / D)
    area = D ** 2

    print('===== 验证 1：d=0 时 lens_in_front_ft == lens_against_ft =====')
    U_ag, x2, y2 = lens_against_ft(tA, wvl, d1, fl)
    U_bf0, _, _ = lens_in_front_ft(tA, wvl, d1, fl, 0.0)
    print('    max|U_ag - U_bf(d=0)| = %.2e（相对 %.2e）'
          % (np.max(np.abs(U_ag - U_bf0)),
             np.max(np.abs(U_ag - U_bf0)) / np.max(np.abs(U_ag))))

    print('===== 验证 2：d=f_l 时退化为精确 FT；与贴透镜差一个二次相位 =====')
    U_bf1, _, _ = lens_in_front_ft(tA, wvl, d1, fl, fl)
    U_exact = 1 / (1j * wvl * fl) * ft2_centered(tA, d1)   # 纯 FT，无相位因子
    ph_dev = np.abs(np.angle(U_bf1 / U_exact))
    print('    max|angle(U_bf(d=f_l)/U_exact)| = %.2e rad（精确 FT，零相位）'
          % ph_dev.max())
    U_exact_ag = U_exact * np.exp(1j * k / (2 * fl) * (x2 ** 2 + y2 ** 2))
    print('    max|U_ag - exp(ik/2f_l r2^2) * U_exact| = %.2e'
          % np.max(np.abs(U_ag - U_exact_ag)))
    print('    焦面二次相位不小儿科：贴透镜时 r2=1 cm 处相位 = %.1f rad'
          % (k / (2 * fl) * 0.01 ** 2))

    print('===== 验证 3：两条路线对照（直接积分 vs FT，物贴透镜） =====')
    # 物面场 = tA * exp(-i k/2f_l (x1^2+y1^2))（透镜紧贴物片之后），
    # 用式 (4.6) 直接传播 Delta_z = f_l；透镜相位与积分的二次相位相消。
    U1 = tA * np.exp(-1j * k / (2 * fl) * (x1 ** 2 + y1 ** 2))
    sl = N // 2
    xs = x2[sl, :]
    U_dir = fresnel_direct_slice(U1, wvl, d1, fl, xs)
    rel = np.max(np.abs(np.abs(U_dir) - np.abs(U_ag[sl, :]))) \
        / np.max(np.abs(U_ag[sl, :]))
    print('    y2=0 切片：max||U_direct|-|U_ft||/max|U_ft| = %.2e' % rel)

    print('===== 验证 4：能量守恒（离散 Parseval） =====')
    d2 = wvl * fl / L
    P_num = np.sum(np.abs(U_ag) ** 2) * d2 ** 2
    print('    sum|U2|^2 d2^2 = %.5e，物面积 = %.5e（相对偏差 %.2e，'
          '来自节点采样把孔径边界样本整格计入/漏计）'
          % (P_num, area, abs(P_num - area) / area))
    print('    轴上辐照度（物贴透镜）= %.6f W/m^2，理论 (D^2/(lambda f_l))^2 = %.6f'
          % (np.abs(U_ag[sl, sl]) ** 2, (area / (wvl * fl)) ** 2))

    print('===== 验证 5：物在透镜后——焦斑宽度与峰值随 d 变化 =====')
    for d in [0.5 * fl, 0.25 * fl]:
        U_bh, x2b, y2b = lens_behind_ft(tA, wvl, d1, d, fl)
        Ib = np.abs(U_bh) ** 2
        row = Ib[N // 2, :]
        peak_i = row.argmax()
        # 实测主瓣半径：峰值右侧第一个 <1% 峰值的样本（sinc^2 主瓣内
        # 只有 >=1% 峰值的台座，sinc(u)^2=0.01 在 u=0.988，误差 <1.2%）
        right = peak_i + np.argmax(row[peak_i:] < 0.01 * row[peak_i])
        x2_axis = x2b[N // 2, :]
        mainlobe = 2 * (x2_axis[right] - x2_axis[peak_i])
        lobe_th = 2 * wvl * d / D          # sinc^2 主瓣全宽 = 2 lambda d / D
        peak_th = (fl / d) ** 2 * (area / (wvl * d)) ** 2
        print('    d=%.2f m：实测峰值 %.4e vs 理论 %.4e；'
              '实测主瓣全宽 %.3e m vs 理论 %.3e m'
              % (d, row.max(), peak_th, mainlobe, lobe_th))

    print('===== 验证 6：渐晕——式 (4.11) 的移位光瞳 vs Listing 4.4 =====')
    # 物前 d=0.5 f_l，有限圆瞳 D_p=2 mm。Listing 4.4 算的是无移位光瞳
    # （相当于 P=1 或"光瞳跟物片重合"）；严格式 (4.11) 对每个 x2 用
    # P(x1 + d/f_l x2)。这里对 y2=0 切片逐点直接求和比较两者。
    d = 0.5 * fl
    Dp = 2e-3
    P0 = circle_p(x1, y1, Dp)                      # 物面处的光瞳（无移位）
    U_no = lens_in_front_ft(tA * P0, wvl, d1, fl, d)
    # 严格版：直接求和 sum tA(x1) P(x1 + d/f_l x2) e^{-i2pi/(lambda f_l) x2 x1}
    inside = tA != 0
    xa, ya = x1[inside], y1[inside]
    xs6 = xs[np.abs(xs) <= 2.5e-3]
    U_strict = np.empty(xs6.size, dtype=complex)
    for j, xv in enumerate(xs6):
        sh = d / fl * xv
        Pv = circle_p(xa + sh, ya, Dp)             # 移位光瞳
        U_strict[j] = np.sum(tA[inside] * Pv
                             * np.exp(-2j * np.pi / (wvl * fl) * xv * xa)) \
            * d1 ** 2 / (1j * wvl * fl) \
            * np.exp(1j * k / (2 * fl) * (1 - d / fl) * xv ** 2)
    U_no_sl = np.abs(U_no[0][sl, :])[np.abs(xs) <= 2.5e-3]
    for xv in [0.0, 1.0e-3, 1.5e-3, 2.0e-3]:
        j = np.argmin(np.abs(xs6 - xv))
        ratio = np.abs(U_strict[j]) / np.abs(U_no_sl[j])
        print('    x2 = %.1f mm：|U_严格(移位光瞳)|/|U_Listing4.4| = %.3f'
              % (xv * 1e3, ratio))
    print('    （主瓣内 |x2|<=1 mm 两者一致——渐晕尚未发生；主瓣外移位光瞳'
          '截断填掉了 sinc 零点，图样定性不同。Listing 4.4 只在 P=1 或'
          '光瞳随物片一起移动时严格成立）')

    if outdir:
        # ---- 图 fig-4-7：三种物位置的焦面辐照度与相位 ----
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
        U_bf05, _, _ = lens_in_front_ft(tA, wvl, d1, fl, 0.5 * fl)
        U_bh, x2b, _ = lens_behind_ft(tA, wvl, d1, 0.5 * fl, fl)
        m5 = np.abs(x2[sl, :]) <= 2.5e-3
        axes[0].plot(x2[sl, :][m5] * 1e3, np.abs(U_ag[sl, :][m5]) ** 2,
                     'k-', lw=1.6, label='物贴透镜（=物前 d=0）')
        axes[0].plot(x2[sl, :][m5] * 1e3, np.abs(U_bf05[sl, :][m5]) ** 2,
                     'b.', ms=5, label='物前 d=0.5$f_l$（辐照度重合）')
        m6 = np.abs(x2b[sl, :]) <= 2.5e-3
        axes[0].plot(x2b[sl, :][m6] * 1e3, np.abs(U_bh[sl, :][m6]) ** 2,
                     'r--', lw=1.4, label='物后 d=0.5$f_l$（标度 $\\lambda d$）')
        axes[0].set_xlabel(r'$x_2$ [mm]')
        axes[0].set_ylabel(r'辐照度 [W/m$^2$]')
        axes[0].set_title('(a) 焦面辐照度：物前位置不影响图样，物后改变标度')
        axes[0].legend(fontsize=9)
        m7 = np.abs(x2[sl, :]) <= 2e-3
        x7 = x2[sl, :][m7]
        x_fine = np.linspace(-2e-3, 2e-3, 2001)
        for d, col, lab in [(0.0, 'k', '物贴透镜（曲率 $1/f_l$）'),
                            (0.5 * fl, 'tab:blue', '物前 d=0.5$f_l$（$0.5/f_l$）'),
                            (fl, 'tab:green', '物前 $d=f_l$（精确 FT，零相位）')]:
            phi = np.angle(np.exp(1j * k / (2 * fl) * (1 - d / fl)
                                  * x_fine ** 2))
            axes[1].plot(x_fine * 1e3, phi, color=col, lw=1.2, label=lab)
        axes[1].plot(x7 * 1e3, np.angle(U_ag[sl, :][m7]), 'k.', ms=6)
        axes[1].plot(x7 * 1e3, np.angle(U_bf05[sl, :][m7]), 'b.', ms=6)
        axes[1].set_xlabel(r'$x_2$ [mm]')
        axes[1].set_ylabel('相位 [rad]')
        axes[1].set_title('(b) 焦面相位记得物的位置（点=数值，线=理论包裹相位）')
        axes[1].legend(fontsize=8, loc='upper center')
        for ax in axes:
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fp = os.path.join(outdir, 'fig-4-7.png')
        fig.savefig(fp, dpi=150)
        print('\n已保存：%s' % fp)
    else:
        plt.show()


def circle_p(X, Y, D):
    """直径 D 的圆瞳（|r| <= D/2 取 1）。"""
    return np.where(np.hypot(X, Y) <= D / 2, 1.0, 0.0)


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)


## 习题

`ch4_problems.py`

In [ ]:
# -*- coding: utf-8 -*-
"""
===== 书 Schmidt PM199 第 4 章 =====
原书 4.3 节习题 1-5：夫琅禾费衍射与透镜焦面的五个数值/解析对比

题目（均按 4.1 节例题的网格 N=512、L=7.5 mm、Dz=20 m、lambda=1 um 复做）：
    1) 1 mm x 1 mm 方孔：解析 U = D^2 sinc(D fx) sinc(D fy)，
       fx = x2/(lambda*Dz)（式 1.67 的方孔谱，见 ch2 笔记 2.6 节）；
    2) 双缝：两个 1 mm 方孔、间隙 0.5 mm（按"边缘间隙"理解，中心距
       1.5 mm；若按中心距 0.5 mm 两孔重叠 0.5 mm，题目失义）。
       解析 = 2 D^2 sinc(D fx) sinc(D fy) cos(2 pi fx x0)，x0=0.75 mm
       ——方孔 sinc 包络 x 双缝余弦条纹；
    3) 1 mm 方孔振幅光栅 tA = 1/2[1+cos(2 pi f0 x)] rect(x/D) rect(y/D)，
       f0 = 10/D：解析 = D^2 sinc(D fy) [1/2 sinc(D fx)
       + 1/4 sinc(D(fx-f0)) + 1/4 sinc(D(fx+f0))]（零级 + 一对一级）；
    4) 1 mm 方孔相位光栅 tA = exp(i 2 pi cos(2 pi f0 x)) rect rect：
       Jacobi-Anger 展开 e^{i z cos t} = sum_n i^n J_n(z) e^{i n t} 给出
       一串衍射级，第 n 级峰值强度 = J_n^2(2 pi) (D^2/(lambda Dz))^2，
       且 sum_n J_n^2(2 pi) = 1（纯相位光栅不吞能量）。
       带宽账本（f0 = 10/D，调制深度 2 pi rad 属于深调制）：
       J_4..J_6 都不小（0.32/0.37/0.28），sum|n|<=3 只装下 31% 能量 ->
       N=512（频带 ±34133 m^-1）会混叠 69%，N=1024 仍混叠 6.2%，
       本题取 N=2048（|n|<=13，占比 100%-6e-9）才干净；
    5) 高斯束聚焦：lambda=1 um、束腰在透镜处、w=2 cm（1/e 振幅半径
       约定，U1 = exp(-r^2/w^2)）、f_l = 1 m、无限孔径。
       解析焦面场 = pi w^2/(i lambda f_l) exp(-(pi w r2/(lambda f_l))^2)
       （高斯的 FT 还是高斯），强度 1/e^2 半径 = lambda f_l/(pi w)。
       网格单独设计：L=0.4 m、N=1024（束宽 2 cm 需要大窗，焦斑 16 um
       需要细 d2 = lambda f_l/L）。

数值验证（运行自动打印）：每题数值 vs 解析的最大偏差、能量守恒、
    习题 4 的 J_n 表与带宽混叠账本、习题 5 的焦斑半径与能量。
MATLAB 注意：Bessel 函数用 besselj(1,pi*x)（jinc）与 besselj(n,2*pi)
    （习题 4）；rect/sinc/jinc 用本地函数（见配套 .m 文件末尾）。
用法：python ch4_problems.py <输出目录>  保存 PNG；否则弹窗显示。
依赖：numpy + matplotlib + scipy（jv）；base.py（sinc/rect/ft2_centered，
    同仓库 python/fourier/common）。
"""
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "..", "fourier", "common"))
from base import rect, sinc, ft2_centered  # noqa: E402


def fraunhofer_prop(Uin, wvl, d1, Dz):
    """原书 Listing 4.1（同 ch4_fraunhofer.py）。"""
    N = Uin.shape[0]
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)
    x2, y2 = np.meshgrid(wvl * Dz * fX, wvl * Dz * fX)
    Uout = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * ft2_centered(Uin, d1)
    return Uout, x2, y2


def lens_against_ft(Uin, wvl, d1, f):
    """原书 Listing 4.3（同 ch4_lenses.py）。"""
    N = Uin.shape[0]
    k = 2 * np.pi / wvl
    fX = (np.arange(N) - N // 2) / (N * d1)
    x2, y2 = np.meshgrid(wvl * f * fX, wvl * f * fX)
    Uout = np.exp(1j * k / (2 * f) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * f) * ft2_centered(Uin, d1)
    return Uout, x2, y2


def err_report(tag, I_num, I_ana):
    """归一化形状误差（各自按峰值归一，扣除孔径离散化的标度差）。"""
    shape = np.abs(I_num / I_num.max() - I_ana / I_ana.max())
    scale = I_num.max() / I_ana.max() - 1
    print('    %s：标度差 %+.2f%%，归一化形状误差 max %.2e'
          % (tag, 100 * scale, shape.max()))
    return shape


def main(outdir=None):
    # ---- 4.1 节例题的公共网格 ----
    N = 512
    L = 7.5e-3
    d1 = L / N
    wvl = 1e-6
    k = 2 * np.pi / wvl
    Dz = 20.0
    n1 = np.arange(N) - N // 2
    x1, y1 = np.meshgrid(n1 * d1, n1 * d1)
    pref = np.exp(1j * k / (2 * Dz) * (x1 ** 2 * 0))  # 占位避免未用告警
    del pref

    def slice_xy(x2, U):
        """取 y2=0 行与对应 x2 升序切片。"""
        sl = U[N // 2, :]
        return x2[N // 2, :], sl

    print('===== 习题 1：1 mm 方孔 =====')
    D = 1e-3
    Uin = rect(x1 / D) * rect(y1 / D)
    Uout, x2, y2 = fraunhofer_prop(Uin, wvl, d1, Dz)
    fx = x2 / (wvl * Dz)
    A1 = D ** 2 * sinc(D * fx) * sinc(D * y2 / (wvl * Dz))
    U_th = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * A1
    xs, us = slice_xy(x2, np.abs(Uout) ** 2)
    _, at = slice_xy(x2, np.abs(U_th) ** 2)
    err_report('习题 1', np.abs(Uout) ** 2, np.abs(U_th) ** 2)
    res1 = (xs, us, at)

    print('===== 习题 2：双缝（1 mm 方孔 x2，边缘间隙 0.5 mm） =====')
    x0 = 0.75e-3
    Uin = (rect((x1 - x0) / D) + rect((x1 + x0) / D)) * rect(y1 / D)
    Uout, x2, y2 = fraunhofer_prop(Uin, wvl, d1, Dz)
    A2 = 2 * D ** 2 * sinc(D * x2 / (wvl * Dz)) \
        * sinc(D * y2 / (wvl * Dz)) * np.cos(2 * np.pi * x2 / (wvl * Dz) * x0)
    U_th = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * A2
    xs, us = slice_xy(x2, np.abs(Uout) ** 2)
    _, at = slice_xy(x2, np.abs(U_th) ** 2)
    err_report('习题 2', np.abs(Uout) ** 2, np.abs(U_th) ** 2)
    res2 = (xs, us, at)

    print('===== 习题 3：振幅光栅（f0 = 10/D） =====')
    f0 = 10 / D
    Uin = 0.5 * (1 + np.cos(2 * np.pi * f0 * x1)) * rect(x1 / D) * rect(y1 / D)
    Uout, x2, y2 = fraunhofer_prop(Uin, wvl, d1, Dz)
    fxg = x2 / (wvl * Dz)
    A3 = D ** 2 * sinc(D * y2 / (wvl * Dz)) * (
        0.5 * sinc(D * fxg)
        + 0.25 * sinc(D * (fxg - f0))
        + 0.25 * sinc(D * (fxg + f0)))
    U_th = np.exp(1j * k / (2 * Dz) * (x2 ** 2 + y2 ** 2)) \
        / (1j * wvl * Dz) * A3
    xs, us = slice_xy(x2, np.abs(Uout) ** 2)
    _, at = slice_xy(x2, np.abs(U_th) ** 2)
    err_report('习题 3', np.abs(Uout) ** 2, np.abs(U_th) ** 2)
    res3 = (xs, us, at)

    print('===== 习题 4：相位光栅（f0 = 10/D，N=2048） =====')
    N4 = 2048
    d14 = L / N4
    n4 = np.arange(N4) - N4 // 2
    x1b, y1b = np.meshgrid(n4 * d14, n4 * d14)
    Uin = np.exp(2j * np.pi * np.cos(2 * np.pi * f0 * x1b)) \
        * rect(x1b / D) * rect(y1b / D)
    Uout, x2b, y2b = fraunhofer_prop(Uin, wvl, d14, Dz)
    I4 = np.abs(Uout) ** 2
    fx4 = x2b / (wvl * Dz)
    nn = np.arange(-8, 9)
    A4 = np.zeros_like(x2b, dtype=complex)
    for n_ord in nn:
        A4 += 1j ** n_ord * jv(n_ord, 2 * np.pi) * D ** 2 \
            * sinc(D * (x2b / (wvl * Dz) - n_ord * f0)) \
            * sinc(D * y2b / (wvl * Dz))
    U_th4 = np.exp(1j * k / (2 * Dz) * (x2b ** 2 + y2b ** 2)) \
        / (1j * wvl * Dz) * A4
    err_report('习题 4', I4, np.abs(U_th4) ** 2)
    scale_pk = (D ** 2 / (wvl * Dz)) ** 2
    print('    各衍射级峰值辐照度（理论 = J_n^2(2pi) x %.4e）：' % scale_pk)
    row = I4[N4 // 2, :]
    fx_row = x2b[N4 // 2, :] / (wvl * Dz)
    for n_ord in [0, 1, 2, 3]:
        j = np.argmin(np.abs(fx_row - n_ord * f0))
        print('      n=%+d：数值 %.4e vs 理论 %.4e（J_%d(2pi)=%.4f）'
              % (n_ord, row[j], jv(n_ord, 2 * np.pi) ** 2 * scale_pk,
                 n_ord, jv(n_ord, 2 * np.pi)))
    s = sum(jv(n_ord, 2 * np.pi) ** 2 for n_ord in range(-40, 41))
    print('    sum_n J_n^2(2pi) = %.6f（=1，纯相位光栅不吞能量）' % s)
    s_all = sum(jv(n_ord, 2 * np.pi) ** 2 for n_ord in range(-60, 61))
    for tag, Nq in [('N=512（原书网格）', 512), ('N=1024', 1024),
                    ('N=2048（本题）', 2048)]:
        nb = int(1 / (2 * (L / Nq) * f0))       # 频带内能装下的最高级数
        s_in = sum(jv(n_ord, 2 * np.pi) ** 2 for n_ord in range(-nb, nb + 1))
        print('    %s：频带 ±%.0f m^-1 装到 |n|<=%d，能量占比 %.1f%%'
              '（混叠 %.2f%%）'
              % (tag, 1 / (2 * (L / Nq)), nb,
                 100 * s_in / s_all, 100 * (1 - s_in / s_all)))
    P4 = np.sum(I4) * (wvl * Dz / L) ** 2
    print('    能量：sum|U2|^2 d2^2 = %.4e vs 物面积 D^2 = %.4e（相对偏差 %.2e）'
          % (P4, D ** 2, abs(P4 - D ** 2) / D ** 2))
    res4 = (x2b[N4 // 2, :], row,
            np.abs(U_th4[N4 // 2, :]) ** 2)

    print('===== 习题 5：高斯束聚焦（w=2 cm，f_l=1 m，无限孔径） =====')
    w = 2e-2
    fl = 1.0
    L5, N5 = 0.4, 1024
    d15 = L5 / N5
    n5 = np.arange(N5) - N5 // 2
    x1c, y1c = np.meshgrid(n5 * d15, n5 * d15)
    Uin5 = np.exp(-(x1c ** 2 + y1c ** 2) / w ** 2)
    Uout5, x2c, y2c = lens_against_ft(Uin5, wvl, d15, fl)
    r2 = np.hypot(x2c, y2c)
    U_th5 = np.pi * w ** 2 / (1j * wvl * fl) \
        * np.exp(-(np.pi * w * r2 / (wvl * fl)) ** 2)
    I5n, I5a = np.abs(Uout5) ** 2, np.abs(U_th5) ** 2
    shape5 = np.abs(I5n / I5n.max() - I5a / I5a.max())
    print('    采样：d1=%.3e m（每 w 跨 %.0f 个样本），d2=%.1e m'
          '（焦斑 1/e^2 半径 %.2e m 跨 %.1f 个样本）'
          % (d15, w / d15, wvl * fl / L5, wvl * fl / (np.pi * w),
             wvl * fl / (np.pi * w) / (wvl * fl / L5)))
    print('    峰值辐照度：数值 %.4e vs 理论 (pi w^2/(lambda f_l))^2 = %.4e'
          % (I5n.max(), (np.pi * w ** 2 / (wvl * fl)) ** 2))
    print('    归一化形状误差 max = %.2e' % shape5.max())
    row5 = I5n[N5 // 2, :]
    x5row = x2c[N5 // 2, :]
    half = row5.max() / np.e ** 2
    i_pk = row5.argmax()
    right = i_pk + np.argmax(row5[i_pk:] < half)
    # 阈值穿点线性内插（相邻样本 2.5 um，直接取样本会有 +0.6 样本量化差）
    xa, xb2 = x5row[right - 1], x5row[right]
    ya, yb2 = row5[right - 1], row5[right]
    x_cross = xb2 - (yb2 - half) / (yb2 - ya) * (xb2 - xa)
    print('    强度 1/e^2 半径：数值 %.4e m vs 理论 lambda f_l/(pi w) = %.4e m'
          % (x_cross - x5row[i_pk], wvl * fl / (np.pi * w)))
    P5 = np.sum(I5n) * (wvl * fl / L5) ** 2
    P5th = np.pi * w ** 2 / 2
    print('    能量：sum|U2|^2 d2^2 = %.4e vs 束总功率 pi w^2/2 = %.4e'
          '（相对偏差 %.2e）' % (P5, P5th, abs(P5 - P5th) / P5th))
    res5 = (x2c[N5 // 2, :], row5, I5a[N5 // 2, :])

    if outdir:
        # ---- 图 fig-4-8：习题 1-4（习题 3/4 的 ±1 级在 x2=±lambda*Dz*f0
        #      = ±200 mm，显示范围放宽才能看到级次结构） ----
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.6))
        titles = ['习题 1：1 mm 方孔',
                  '习题 2：双缝（间隙 0.5 mm）',
                  '习题 3：振幅光栅 $f_0=10/D$',
                  '习题 4：相位光栅 $f_0=10/D$（N=2048）']
        halves = [0.1, 0.1, 0.3, 0.7]
        for ax, res, ti, hw in zip(axes.ravel(),
                                   [res1, res2, res3, res4], titles, halves):
            xs, us, at = res
            m = np.abs(xs) <= hw
            ax.plot(xs[m] * 1e3, at[m], color='0.6', lw=1.6, ls='--',
                    label='解析')
            ax.plot(xs[m] * 1e3, us[m], 'b.', ms=3, label='数值')
            ax.set_xlabel(r'$x_2$ [mm]')
            ax.set_ylabel(r'辐照度 [W/m$^2$]')
            ax.set_title(ti, fontsize=11)
            ax.legend(fontsize=9)
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fp = os.path.join(outdir, 'fig-4-8.png')
        fig.savefig(fp, dpi=150)
        print('\n已保存：%s' % fp)

        # ---- 图 fig-4-9：习题 5 ----
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
        msrc = np.abs(x1c[0, :]) <= 0.05
        axes[0].plot(x1c[0, :][msrc], Uin5[N5 // 2, :][msrc], 'k-', lw=1.4)
        axes[0].axvline(-w, color='r', ls=':', lw=1)
        axes[0].axvline(w, color='r', ls=':', lw=1)
        axes[0].set_xlabel(r'$x_1$ [m]')
        axes[0].set_ylabel('振幅')
        axes[0].set_title('(a) 透镜面振幅（红线 $\\pm w$，窗 L=0.4 m 截断'
                          ' $e^{-100}$）')
        mf = np.abs(res5[0]) <= 6e-5
        axes[1].plot(res5[0][mf] * 1e6, res5[2][mf], color='0.6', lw=1.8,
                     ls='--', label='解析')
        axes[1].plot(res5[0][mf] * 1e6, res5[1][mf], 'b.', ms=4, label='数值')
        axes[1].set_xlabel(r'$x_2$ [$\mu$m]')
        axes[1].set_ylabel(r'辐照度 [W/m$^2$]')
        axes[1].set_title('(b) 焦面光斑（1/e$^2$ 半径 15.9 $\\mu$m）')
        axes[1].legend(fontsize=9)
        for ax in axes:
            ax.grid(alpha=0.25)
        fig.tight_layout()
        fp = os.path.join(outdir, 'fig-4-9.png')
        fig.savefig(fp, dpi=150)
        print('已保存：%s' % fp)
    else:
        plt.show()


if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else None)
